# F1 Prediction Platform — Clean Recovery Workflow

This notebook rebuilds the modelling workflow after a kernel restart using the saved processed dataset. It does **not** recollect data from the F1 API.

**Run cells from top to bottom, in order.** If a cell fails, stop there and share the complete error before running later cells.

Project root expected:
`C:\Users\saksham\Desktop\Project\f1-prediction-platform`

This is a clean recovery workflow to avoid relying on variables left in memory by `01_data_collection.ipynb`.


In [8]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

PROJECT_DIR = Path(r"C:\Users\saksham\Desktop\Project\f1-prediction-platform")
DATA_PATH = PROJECT_DIR / "data" / "processed" / "expanded_model_dataset.csv"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder exists:", PROJECT_DIR.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)
print("Models folder:", MODEL_DIR)
print("Reports folder:", REPORT_DIR)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        "Project folder was not found. Update PROJECT_DIR to your actual project path."
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "expanded_model_dataset.csv was not found. Do not recollect data yet; "
        "check the project's data/processed folder."
    )


Project folder exists: True
Dataset exists: True
Dataset path: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\processed\expanded_model_dataset.csv
Models folder: C:\Users\saksham\Desktop\Project\f1-prediction-platform\models
Reports folder: C:\Users\saksham\Desktop\Project\f1-prediction-platform\reports


In [9]:
# Load the saved historical dataset
expanded_df = pd.read_csv(DATA_PATH)

expanded_features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]
baseline_features = ["starting_position_feature"]
target = "won_race"

required_columns = [
    "season", "round", "race_name", "driver_name", "driver_id",
    "constructor_name", target, *expanded_features
]
missing_columns = [col for col in required_columns if col not in expanded_df.columns]

print("Dataset shape:", expanded_df.shape)
print("Seasons:", sorted(expanded_df["season"].dropna().astype(int).unique()))
print("Missing required columns:", missing_columns)

if missing_columns:
    raise ValueError(
        "The saved dataset is missing required columns. "
        "Share this list before running further cells."
    )

expanded_df["season"] = pd.to_numeric(expanded_df["season"], errors="coerce")
expanded_df[target] = pd.to_numeric(expanded_df[target], errors="coerce")

expanded_train = expanded_df[expanded_df["season"].between(2018, 2024)].copy()
expanded_test = expanded_df[expanded_df["season"] == 2025].copy()

if expanded_train.empty or expanded_test.empty:
    raise ValueError("Expected training seasons 2018–2024 and test season 2025.")

print("Training rows:", len(expanded_train))
print("Test rows:", len(expanded_test))
print("Training seasons:", sorted(expanded_train["season"].unique()))
print("Test seasons:", sorted(expanded_test["season"].unique()))

winner_counts = expanded_test.groupby(["season", "round"])[target].sum()
print("2025 races:", len(winner_counts))
print("Races with exactly one winner:", int((winner_counts == 1).sum()))
print("Races with invalid winner counts:", int((winner_counts != 1).sum()))

if not (winner_counts == 1).all():
    raise ValueError("At least one 2025 race does not have exactly one winner.")


Dataset shape: (3458, 27)
Seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Missing required columns: []
Training rows: 2979
Test rows: 479
Training seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Test seasons: [np.int64(2025)]
2025 races: 24
Races with exactly one winner: 24
Races with invalid winner counts: 0


In [10]:
# Train reproducible models from the saved dataset.
# This intentionally rebuilds the models so no old in-memory variables are needed.

baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])

logistic_history_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])

random_forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=10,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )),
])

X_train_baseline = expanded_train[baseline_features]
X_test_baseline = expanded_test[baseline_features]
X_train_history = expanded_train[expanded_features]
X_test_history = expanded_test[expanded_features]
y_train = expanded_train[target].astype(int)

baseline_model.fit(X_train_baseline, y_train)
logistic_history_model.fit(X_train_history, y_train)
random_forest_model.fit(X_train_history, y_train)

joblib.dump(baseline_model, MODEL_DIR / "baseline_model.joblib")
joblib.dump(logistic_history_model, MODEL_DIR / "logistic_history_model.joblib")
joblib.dump(random_forest_model, MODEL_DIR / "random_forest_model.joblib")

print("Models trained and saved:")
print("-", MODEL_DIR / "baseline_model.joblib")
print("-", MODEL_DIR / "logistic_history_model.joblib")
print("-", MODEL_DIR / "random_forest_model.joblib")


Models trained and saved:
- C:\Users\saksham\Desktop\Project\f1-prediction-platform\models\baseline_model.joblib
- C:\Users\saksham\Desktop\Project\f1-prediction-platform\models\logistic_history_model.joblib
- C:\Users\saksham\Desktop\Project\f1-prediction-platform\models\random_forest_model.joblib


In [11]:
# Rebuild the 2025 driver-level predictions and race rankings
test_predictions = expanded_test[
    [
        "season", "round", "race_name", "driver_id", "driver_name",
        "constructor_name", "grid", "starting_position_feature", target
    ]
].copy()

test_predictions["baseline_score"] = baseline_model.predict_proba(
    X_test_baseline
)[:, 1]

test_predictions["logistic_score"] = logistic_history_model.predict_proba(
    X_test_history
)[:, 1]

test_predictions["random_forest_score"] = random_forest_model.predict_proba(
    X_test_history
)[:, 1]

for score_col, rank_col in [
    ("baseline_score", "baseline_rank"),
    ("logistic_score", "logistic_rank"),
    ("random_forest_score", "rf_rank"),
]:
    test_predictions[rank_col] = (
        test_predictions.groupby(["season", "round"])[score_col]
        .rank(method="first", ascending=False)
        .astype(int)
    )

def summarize_rankings(df, rank_col, model_name):
    winners = df[df[target] == 1]
    return {
        "model": model_name,
        "races_evaluated": winners[["season", "round"]].drop_duplicates().shape[0],
        "top1_correct": int((winners[rank_col] == 1).sum()),
        "top1_accuracy_pct": round((winners[rank_col] == 1).mean() * 100, 2),
        "winner_in_top3_pct": round((winners[rank_col] <= 3).mean() * 100, 2),
        "mean_actual_winner_rank": round(winners[rank_col].mean(), 2),
    }

model_comparison = pd.DataFrame([
    summarize_rankings(test_predictions, "baseline_rank", "Starting-grid baseline"),
    summarize_rankings(test_predictions, "logistic_rank", "Logistic Regression + history"),
    summarize_rankings(test_predictions, "rf_rank", "Random Forest + history"),
])

display(model_comparison)

prediction_path = REPORT_DIR / "predictions_2025_rebuilt.csv"
comparison_path = REPORT_DIR / "model_comparison_2025_rebuilt.csv"
test_predictions.to_csv(prediction_path, index=False)
model_comparison.to_csv(comparison_path, index=False)

print("Saved predictions:", prediction_path)
print("Saved model comparison:", comparison_path)


,model,races_evaluated,top1_correct,top1_accuracy_pct,winner_in_top3_pct,mean_actual_winner_rank
0,Starting-grid baseline,24,16,66.67,95.83,1.54
1,Logistic Regression + history,24,12,50.00,100.00,1.67
2,Random Forest + history,24,14,58.33,87.50,1.79


Saved predictions: C:\Users\saksham\Desktop\Project\f1-prediction-platform\reports\predictions_2025_rebuilt.csv
Saved model comparison: C:\Users\saksham\Desktop\Project\f1-prediction-platform\reports\model_comparison_2025_rebuilt.csv


In [12]:
# Normalize each model's positive-class scores within each race.
# This makes the scores sum to 1 per race; it does not by itself calibrate probabilities.

score_columns = {
    "Baseline": "baseline_score",
    "Logistic Regression": "logistic_score",
    "Random Forest": "random_forest_score",
}

normalized_predictions = test_predictions.copy()
probability_columns = {}

for model_name, score_col in score_columns.items():
    probability_col = {
        "Baseline": "baseline_normalized_probability",
        "Logistic Regression": "logistic_normalized_probability",
        "Random Forest": "rf_normalized_probability",
    }[model_name]

    score_sums = normalized_predictions.groupby(
        ["season", "round"]
    )[score_col].transform("sum")

    if (score_sums <= 0).any():
        raise ValueError(f"Non-positive score sum found for {model_name}.")

    normalized_predictions[probability_col] = (
        normalized_predictions[score_col] / score_sums
    )
    probability_columns[model_name] = probability_col

probability_sums = normalized_predictions.groupby(["season", "round"])[
    list(probability_columns.values())
].sum()

print("Probability sums by race (should all be 1.0):")
display(probability_sums.head())

print("Maximum absolute deviation from 1:")
display((probability_sums - 1).abs().max())

print("Actual winner probability examples:")
display(
    normalized_predictions[normalized_predictions[target] == 1][
        ["round", "race_name", "driver_name", *probability_columns.values()]
    ].sort_values("round").head(12)
)


Probability sums by race (should all be 1.0):


baseline_normalized_probability  \
season round                                    
2025   1                                  1.0   
       2                                  1.0   
       3                                  1.0   
       4                                  1.0   
       5                                  1.0   

              logistic_normalized_probability  rf_normalized_probability  
season round                                                              
2025   1                                  1.0                        1.0  
       2                                  1.0                        1.0  
       3                                  1.0                        1.0  
       4                                  1.0                        1.0  
       5                                  1.0                        1.0

Maximum absolute deviation from 1:


baseline_normalized_probability    2.220446e-16
logistic_normalized_probability    2.220446e-16
rf_normalized_probability          2.220446e-16
dtype: float64

Actual winner probability examples:


,round,race_name,driver_name,baseline_normalized_probability,logistic_normalized_probability,rf_normalized_probability
2992,1,Australian Grand Prix,Lando Norris,0.369615,0.298782,0.332296
3014,2,Chinese Grand Prix,Oscar Piastri,0.369615,0.189612,0.219008
3031,3,Japanese Grand Prix,Max Verstappen,0.369615,0.472608,0.406242
3054,4,Bahrain Grand Prix,Oscar Piastri,0.369615,0.356162,0.260746
3074,5,Saudi Arabian Grand Prix,Oscar Piastri,0.248291,0.234978,0.180433
3094,6,Miami Grand Prix,Oscar Piastri,0.094874,0.154455,0.063698
3111,7,Emilia Romagna Grand Prix,Max Verstappen,0.248291,0.330637,0.300151
3132,8,Monaco Grand Prix,Lando Norris,0.369615,0.305044,0.269737
3154,9,Spanish Grand Prix,Oscar Piastri,0.369619,0.359153,0.268014
3174,10,Canadian Grand Prix,George Russell,0.369615,0.123520,0.241977


In [13]:
# Evaluate normalized race-level probabilities against a uniform baseline
evaluation_rows = []

for (season, round_no), race in normalized_predictions.groupby(["season", "round"]):
    actual = race[target].to_numpy(dtype=float)
    n_drivers = len(race)

    for model_name, probability_col in probability_columns.items():
        probabilities = race[probability_col].to_numpy(dtype=float)
        probabilities = np.clip(probabilities, 1e-15, 1.0)
        probabilities = probabilities / probabilities.sum()

        winner_probability = probabilities[actual == 1][0]
        evaluation_rows.append({
            "season": season,
            "round": round_no,
            "model": model_name,
            "winner_probability": winner_probability,
            "log_loss": -np.log(winner_probability),
            "brier_score": np.sum((probabilities - actual) ** 2),
        })

normalized_metrics = pd.DataFrame(evaluation_rows)

model_metrics = (
    normalized_metrics.groupby("model")
    .agg(
        races_evaluated=("round", "count"),
        mean_log_loss=("log_loss", "mean"),
        mean_brier_score=("brier_score", "mean"),
        mean_winner_probability=("winner_probability", "mean"),
    )
    .reset_index()
)

uniform_rows = []
for (season, round_no), race in normalized_predictions.groupby(["season", "round"]):
    n_drivers = len(race)
    actual = race[target].to_numpy(dtype=float)
    uniform_probability = np.full(n_drivers, 1 / n_drivers)
    uniform_rows.append({
        "season": season,
        "round": round_no,
        "log_loss": np.log(n_drivers),
        "brier_score": np.sum((uniform_probability - actual) ** 2),
        "winner_probability": 1 / n_drivers,
    })

uniform_df = pd.DataFrame(uniform_rows)
uniform_summary = pd.DataFrame([{
    "model": "Uniform baseline",
    "races_evaluated": len(uniform_df),
    "mean_log_loss": uniform_df["log_loss"].mean(),
    "mean_brier_score": uniform_df["brier_score"].mean(),
    "mean_winner_probability": uniform_df["winner_probability"].mean(),
}])

model_metrics = pd.concat([model_metrics, uniform_summary], ignore_index=True)
model_metrics = model_metrics.sort_values("mean_log_loss").round(4)

display(model_metrics)

probability_path = REPORT_DIR / "race_level_probabilities_2025_normalized.csv"
metrics_path = REPORT_DIR / "probability_evaluation_2025_normalized.csv"

normalized_predictions.to_csv(probability_path, index=False)
model_metrics.to_csv(metrics_path, index=False)

print("Saved race-level probabilities:", probability_path)
print("Saved probability evaluation:", metrics_path)
print("\nRecovery workflow completed successfully.")


,model,races_evaluated,mean_log_loss,mean_brier_score,mean_winner_probability
1,Logistic Regression,24,1.1693,0.5980,0.3442
0,Baseline,24,1.2254,0.6138,0.3114
2,Random Forest,24,1.3708,0.6585,0.2878
3,Uniform baseline,24,2.9936,0.9499,0.0501


Saved race-level probabilities: C:\Users\saksham\Desktop\Project\f1-prediction-platform\reports\race_level_probabilities_2025_normalized.csv
Saved probability evaluation: C:\Users\saksham\Desktop\Project\f1-prediction-platform\reports\probability_evaluation_2025_normalized.csv

Recovery workflow completed successfully.


In [14]:

models_to_check = {
    "Baseline": baseline_model,
    "Logistic Regression": logistic_history_model,
    "Random Forest": random_forest_model,
}

for name, model in models_to_check.items():
    print(f"{name}: classes_ = {model.classes_}")


Baseline: classes_ = [0 1]
Logistic Regression: classes_ = [0 1]
Random Forest: classes_ = [0 1]


In [15]:

models_to_check = {
    "Baseline": baseline_model,
    "Logistic Regression": logistic_history_model,
    "Random Forest": random_forest_model,
}

for name, model in models_to_check.items():
    print(f"\n{name}")

    # Works with the saved sklearn Pipeline structure
    classifier = model.steps[-1][1]

    print("Classifier:", type(classifier).__name__)
    print("class_weight:", getattr(classifier, "class_weight", None))



Baseline
Classifier: LogisticRegression
class_weight: None

Logistic Regression
Classifier: LogisticRegression
class_weight: None

Random Forest
Classifier: RandomForestClassifier
class_weight: None


In [16]:

for name, model in models_to_check.items():
    print(f"\n{'=' * 50}")
    print(name)

    classifier = model.steps[-1][1]
    print("Classifier parameters:")
    print({
        key: value
        for key, value in classifier.get_params().items()
        if key in [
            "class_weight",
            "C",
            "max_iter",
            "solver",
            "n_estimators",
            "max_depth",
            "random_state",
        ]
    })



Baseline
Classifier parameters:
{'C': 1.0, 'class_weight': None, 'max_iter': 2000, 'random_state': 42, 'solver': 'lbfgs'}

Logistic Regression
Classifier parameters:
{'C': 1.0, 'class_weight': None, 'max_iter': 2000, 'random_state': 42, 'solver': 'lbfgs'}

Random Forest
Classifier parameters:
{'class_weight': None, 'max_depth': None, 'n_estimators': 400, 'random_state': 42}
